# Chapter 12: Fine-Tuning a Generative Model with LoRA

Full fine-tuning of a 7B parameter model requires ~112 GB of GPU memory. **LoRA** (Low-Rank Adaptation) fixes this by training only a tiny set of *adapter* weights injected into the model.

```
Original weight W (frozen)  +  low-rank adapter ΔW = W + BA
```
Typically ΔW adds <1% extra parameters.

## Setup

In [ ]:
# !pip install peft transformers datasets torch bitsandbytes
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer
print('Ready')

## Step 1 — Load base model

In [ ]:
MODEL = "facebook/opt-125m"   # tiny model for demo — swap for "meta-llama/Llama-2-7b" for real use
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)
print(f"Base parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 2 — Apply LoRA

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                  # rank — higher = more expressive but more params
    lora_alpha=16,        # scaling factor
    target_modules=["q_proj", "v_proj"],   # which layers to adapt
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Typical output: trainable params: ~300k  ||  all params: 125M  ||  trainable%: 0.24%

## Step 3 — Prepare training data

In [ ]:
from datasets import Dataset

# Simple instruction-following examples
examples = [
    {"text": "### Instruction: Summarise in one sentence.\n### Input: The Eiffel Tower was built in 1889 in Paris, France. It is 330 metres tall and was designed by Gustave Eiffel.\n### Response: The Eiffel Tower is a 330-metre landmark in Paris built by Gustave Eiffel in 1889."},
    {"text": "### Instruction: Convert to passive voice.\n### Input: The chef cooked the meal.\n### Response: The meal was cooked by the chef."},
    {"text": "### Instruction: Translate to French.\n### Input: Good morning, how are you?\n### Response: Bonjour, comment allez-vous?"},
    {"text": "### Instruction: Explain like I am five.\n### Input: What is gravity?\n### Response: Gravity is an invisible force that pulls things toward each other. It is why you fall down and not up when you jump!"},
]

def tokenize(ex):
    return tokenizer(ex["text"], truncation=True, padding="max_length", max_length=256, return_tensors=None)

ds = Dataset.from_list(examples).map(tokenize, batched=False)
ds = ds.add_column("labels", ds["input_ids"])
ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print(f"Dataset: {len(ds)} examples")

## Step 4 — Train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir="/tmp/opt-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-4,
    fp16=False,      # set True if you have a GPU
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()
print("Training complete")

## Step 5 — Save and load the adapter

In [ ]:
model.save_pretrained("/tmp/opt-lora-adapter")
print("Adapter saved to /tmp/opt-lora-adapter")

# To reload:
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained(MODEL)
# model = PeftModel.from_pretrained(base, "/tmp/opt-lora-adapter")

## Step 6 — Generate with the fine-tuned adapter

In [ ]:
from transformers import pipeline

gen = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=80)

prompt = "### Instruction: Summarise in one sentence.\n### Input: "
prompt += "Python is an interpreted, high-level, general-purpose programming language. Created by Guido van Rossum, it was first released in 1991."
prompt += "\n### Response:"

output = gen(prompt)[0]["generated_text"]
print(output[len(prompt):].strip())

## Summary

| Technique | Trainable params | Memory |
|-----------|-----------------|--------|
| Full fine-tune | 100% | Very high |
| LoRA (r=8) | ~0.2–1% | Low |
| QLoRA (4-bit) | ~0.2–1% | Very low |

For a 7B model on a 24 GB GPU: full fine-tune = impossible, QLoRA = feasible. LoRA is the standard way to fine-tune LLMs in 2024–2025.